In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = ""

In [30]:
!pip install -q -U \
    youtube-transcript-api \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-google-genai \
    sentence-transformers \
    faiss-cpu \
    gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 19.8 MB/s eta 0:00:00


In [31]:
import os

from urllib.parse import urlparse, parse_qs

from youtube_transcript_api import (
    YouTubeTranscriptApi,
    TranscriptsDisabled
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_huggingface import (
    HuggingFaceEmbeddings
)

from langchain_community.vectorstores import (
    FAISS
)

from langchain_google_genai import (
    ChatGoogleGenerativeAI
)

from langchain_core.prompts import (
    PromptTemplate
)

import gradio as gr

In [32]:
def extract_video_id(url):

    parsed_url = urlparse(url)

    # youtube.com/watch?v=...
    if parsed_url.hostname in [
        "www.youtube.com",
        "youtube.com"
    ]:

        return parse_qs(
            parsed_url.query
        ).get("v", [None])[0]

    # youtu.be/...
    if parsed_url.hostname == "youtu.be":

        return parsed_url.path.lstrip("/")

    return None

In [33]:
url = "https://www.youtube.com/watch?v=Gfr50f6ZBvo"

video_id = extract_video_id(url)

print(video_id)

Gfr50f6ZBvo


In [34]:
def get_transcript(video_id):

    try:

        api = YouTubeTranscriptApi()

        transcript_data = api.fetch(
            video_id,
            languages=["en"]
        )

        transcript_list = (
            transcript_data.to_raw_data()
        )

        transcript = " ".join(
            chunk["text"]
            for chunk in transcript_list
        )

        return transcript

    except TranscriptsDisabled:

        return None

    except Exception as e:

        print("Transcript error:", e)

        return None

In [35]:
transcript = get_transcript(
    "Gfr50f6ZBvo"
)

print(transcript[:1000])

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough 

In [36]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [37]:
chunks = text_splitter.create_documents(
    [transcript]
)

print("Number of chunks:", len(chunks))

Number of chunks: 168


In [38]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [40]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

In [41]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [43]:
retriever.invoke("What is deepmind")

[Document(id='eec0b7fd-4219-453c-b9be-829abb9d131c', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

In [44]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [45]:
prompt = PromptTemplate(
    template="""
You are an intelligent YouTube video assistant.

Answer the user's question using ONLY
the information provided in the transcript context.

Rules:

1. Do not invent information.
2. Do not use outside knowledge.
3. If the answer is not available in the
   transcript, say:
   "I couldn't find the answer in the video."
4. Give a clear and concise answer.

Transcript Context:
{context}

User Question:
{question}

Answer:
""",

    input_variables=[
        "context",
        "question"
    ]
)

In [46]:
def ask_question(question):

    if not question.strip():

        return "Please enter a question."

    # Retrieve relevant chunks
    docs = retriever.invoke(question)

    # Combine chunks
    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    # Create prompt
    final_prompt = prompt.format(
        context=context,
        question=question
    )

    # Ask Gemini
    response = llm.invoke(
        final_prompt
    )

    return response.content

In [47]:
question = "What is the main topic of the video?"

answer = ask_question(question)

print(answer)

The main topic of the video appears to be the challenge of solving intelligence, particularly in the context of DeepMind's work, and understanding the human mind and the universe through science and technology, including discussions on topics like consciousness, life, gravity, and protein folding (AlphaFold).
